# 03 — Chapter IV assets

Everything Chapter IV needs that cannot be produced from the analysis bundle:
per-view metrics, the render figures for all ten configurations, and four data
products that need the full run directories rather than the 48 diagnostics the
bundle carries.

Runs against the finished campaign. It trains nothing and writes nothing into
`runs/` — it reads checkpoints and writes a folder you download.

**Order matters.** The cheap file-reading products come first, so a disconnect
during the long pass still leaves you with something. The self-check in
section 7 is the gate: do not use the per-view numbers until it reads `ok` on
every run.

Expect 45–70 minutes on an A100 for a complete run — most of it LPIPS over
held-out views, and 240 renders.

## 1. Mount Drive and set the campaign root

The same header every other notebook uses: it defines `DRIVE_ROOT` and the
repository constants the clone cell needs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ---------------------------------------------------------------------------
# The one place paths are defined. Everything else derives from DRIVE_ROOT.
#
#   e3dgsuw/
#     dataset/     the four scenes (original) + undistorted/  <- created below
#     dense/       M1 clouds, with SHA-256 sidecars
#     runs/        <cell>/<scene>/s<seed>/  -- one run, all of it together
#     analysis/    analyse.py output, figures, tables
#     run_ledger.json
# ---------------------------------------------------------------------------
DRIVE_ROOT   = '/content/drive/MyDrive/e3dgsuw'
DATASET_DIR  = f'{DRIVE_ROOT}/dataset'
DATA_UNDIST  = f'{DATASET_DIR}/undistorted'
DENSE_DIR    = f'{DRIVE_ROOT}/dense'
ANALYSIS_DIR = f'{DRIVE_ROOT}/analysis'

# Training reads from local disk, not Drive: the scene loader pulls every image
# at startup, and Drive's FUSE layer makes that far slower than a single copy.
LOCAL_DATA   = '/content/data'

REPO_URL  = 'https://github.com/dinanirham/An-Efficient-3D-Gaussian-Splatting-for-Underwater-3D-Reconstruction.git'
REPO_DIR  = '/content/e3dgsuw'
IMPL_DIR  = f'{REPO_DIR}/implementation'
SCENES    = ['Curasao', 'IUI3-RedSea', 'JapaneseGradens-RedSea', 'Panama']

import os
assert os.path.isdir(DRIVE_ROOT), (
    f'{DRIVE_ROOT} not found. Check the folder name, or edit DRIVE_ROOT above.')
for d in (DATA_UNDIST, DENSE_DIR, f'{DRIVE_ROOT}/runs', ANALYSIS_DIR):
    os.makedirs(d, exist_ok=True)

# Export them so the `!` cells below resolve "$DRIVE_ROOT" as a real shell
# variable. Relying on IPython to substitute notebook variables into magics
# works until it doesn't, and when it doesn't it substitutes nothing and the
# command runs against a silently truncated path rather than failing.
os.environ.update(
    DRIVE_ROOT=DRIVE_ROOT, DATASET_DIR=DATASET_DIR, DATA_UNDIST=DATA_UNDIST,
    DENSE_DIR=DENSE_DIR, ANALYSIS_DIR=ANALYSIS_DIR, LOCAL_DATA=LOCAL_DATA,
    REPO_DIR=REPO_DIR, IMPL_DIR=IMPL_DIR,
)


def find_originals():
    """Locate the four scenes under dataset/, however they were arranged.

    Accepts the scenes directly under dataset/, or nested one level (e.g.
    dataset/SeathruNeRF_dataset/). Returns the directory that contains them.
    """
    candidates = [DATASET_DIR] + [
        os.path.join(DATASET_DIR, d) for d in sorted(os.listdir(DATASET_DIR))
        if os.path.isdir(os.path.join(DATASET_DIR, d)) and d != 'undistorted'
    ]
    for base in candidates:
        if all(os.path.isdir(os.path.join(base, s)) for s in SCENES):
            return base
    return None


DATA_ORIG = find_originals()

# verify_undistort's T1 -- the check that would catch the undistortion gap --
# reads the *original* dataset. Point it at wherever it actually landed on
# Drive, or T1 reports "dataset not found" and the one check that matters here
# quietly stops testing anything.
if DATA_ORIG:
    os.environ['E3DGSUW_DATASET'] = DATA_ORIG

print('drive root :', DRIVE_ROOT)
print('originals  :', DATA_ORIG or 'NOT FOUND')
print('undistorted:', DATA_UNDIST)


In [ ]:
ASSETS = '/content/ch4_assets'

import os
assert os.path.isdir(f'{DRIVE_ROOT}/runs'), f'{DRIVE_ROOT}/runs not found'
print('campaign root :', DRIVE_ROOT)
print('cells present :', sorted(os.listdir(f'{DRIVE_ROOT}/runs')))
print('writing to    :', ASSETS)

## 2. Clone and build

This notebook is meant to run on a fresh runtime — it does not assume
`00_setup` has been run here, because collecting assets from a finished
campaign has no reason to wait on dataset preparation.

The build is needed from section 5 onward, which renders.

In [ ]:
import os, subprocess

# Private repo? Add a Colab secret named GITHUB_TOKEN (key icon in the left
# sidebar) with a fine-grained read token, and toggle notebook access on.
# Read from Secrets rather than pasted into the cell: a pasted token is saved
# inside the .ipynb, which then travels wherever the notebook does.
GITHUB_TOKEN = None
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN') or None
    print('GITHUB_TOKEN: loaded from Colab Secrets')
except ImportError:
    pass                                  # not running under Colab
except Exception as e:                    # secret absent, or access not granted
    print(f'GITHUB_TOKEN: not available ({type(e).__name__}) -- '
          'fine for a public repo')

url = REPO_URL
if GITHUB_TOKEN:
    url = REPO_URL.replace('https://', f'https://{GITHUB_TOKEN}@')

# Never let git fall back to an interactive credential prompt: in a notebook it
# hangs the cell indefinitely with nothing on screen to say why.
env = {**os.environ, 'GIT_TERMINAL_PROMPT': '0'}


def _redact(s):
    """Strip the token from git output -- git echoes the remote URL on failure,
    and notebook outputs are saved to the file and shared with it."""
    return s.replace(GITHUB_TOKEN, '***') if GITHUB_TOKEN else s


if os.path.isdir(REPO_DIR) and not os.path.isdir(f'{REPO_DIR}/.git'):
    raise RuntimeError(
        f'{REPO_DIR} exists but is not a git checkout -- probably a clone that '
        f'died partway. Delete it and re-run this cell.')

if not os.path.exists(REPO_DIR):
    r = subprocess.run(['git','clone','--depth','1',url,REPO_DIR],
                       capture_output=True, text=True, env=env)
    if r.returncode != 0:
        raise RuntimeError(
            'clone failed. If the repository is private, add a GITHUB_TOKEN '
            'secret in Colab and grant this notebook access.\n'
            f'{_redact(r.stderr)[-800:]}')
else:
    # Repoint the remote before pulling. The stored URL was written by an
    # earlier clone, which may have run without a token (or with a stale one);
    # injecting the token into `url` alone never reaches the pull.
    subprocess.run(['git','-C',REPO_DIR,'remote','set-url','origin',url],
                   check=True, env=env)
    r = subprocess.run(['git','-C',REPO_DIR,'pull','--ff-only'],
                       capture_output=True, text=True, env=env)
    if r.returncode != 0:
        raise RuntimeError(
            'pull failed. If the repository is private, check the GITHUB_TOKEN '
            'secret is set and this notebook has access.\n'
            f'{_redact(r.stderr)[-800:]}')

# Fail here, naming the directory, rather than letting a later cell run from
# whatever the working directory happened to be.
assert os.path.isdir(IMPL_DIR), (
    f'clone produced no {IMPL_DIR}. Contents of {REPO_DIR}: '
    f'{sorted(os.listdir(REPO_DIR)) if os.path.isdir(REPO_DIR) else "missing"}')

os.chdir(IMPL_DIR)
print(subprocess.run(['git','-C',REPO_DIR,'log','--oneline','-1'],
                     capture_output=True, text=True).stdout.strip())
print('cwd:', os.getcwd())


In [ ]:
# Builds diff_gaussian_rasterization_ms and simple_knn against whatever torch
# Colab ships -- deliberately NOT installing our own, which would risk a
# mismatch between torch's CUDA and the toolkit the extensions compile with.
# Takes a few minutes; must be repeated each session.
#
# chdir explicitly rather than via `%cd $IMPL_DIR`: a magic whose variable fails
# to expand reports the *current* directory and continues, so the build then
# runs from the wrong place and fails two steps later with a bare
# "tools/setup_colab.sh: No such file or directory".
import os
assert os.path.isdir(IMPL_DIR), (
    f'{IMPL_DIR} not found -- run the "Clone the repository" cell above first.')
os.chdir(IMPL_DIR)
print('building in', os.getcwd())
!bash tools/setup_colab.sh


In [ ]:
import importlib, torch
for m in ('diff_gaussian_rasterization_ms', 'simple_knn'):
    importlib.import_module(m)
print('extensions import OK  |  torch', torch.__version__,
      '| cuda', torch.version.cuda)


In [ ]:
%cd /content/e3dgsuw/implementation
!pip install -q plyfile
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 3. Check the collector before spending GPU time

Twelve tests over the pure parts: view selection, crop arithmetic, the
self-check thresholds, and the rule that the compressed store has exactly one
reader. They take a second and catch the errors that would otherwise surface
forty minutes into a render pass.

In [ ]:
!python tools/verify_chapter_assets.py

## 4. The file-reading products

C4, C5 and C6 read diagnostics and point clouds rather than rendering, so they
finish in about a minute:

* **C4** per-frame depth-range distributions — the quantity the registered
  explanation was about
* **C5** medium parameters over training for all 120 runs; the analysis bundle
  carried only the 48 simplification runs
* **C6** primitive distance from the cloud centre, the evidence for the
  invisible-population account

In [ ]:
!python -m tools.chapter_assets \
    --output_root "$DRIVE_ROOT" --out "$ASSETS" --only C4,C5,C6

## 5. Per-view metrics and the renders — the long pass

Loads each seed-0 checkpoint, renders every held-out view, and records the four
metrics per image under the campaign's own conventions rather than a second
implementation of them. Renders all ten configurations at one fixed view and
crop per scene, so the qualitative figures can put ground truth, the reference
and the baseline beside every mechanism.

**Quantised runs are rendered in their codebook state.** The stored point cloud
holds the *continuous* parameters, which the campaign never evaluated; an
earlier version of this collector measured those by mistake and reported
quantisation as costing six decibels.

**Bounded by the storage policy.** Full point clouds are kept for seed 0, so
per-view metrics cover one repeat per cell and scene. That answers whether a
scene mean rests on a single bad view; it cannot show how per-view fidelity
moves between repeats.

In [ ]:
!python -m tools.chapter_assets \
    --output_root "$DRIVE_ROOT" --out "$ASSETS" --only C1,renders

## 6. The two extra render passes

* **The invisible population.** The baseline is rendered twice: as it stands,
  and with every primitive below the visibility threshold silenced rather than
  deleted, so the second render is provably the same model minus a subset.
* **Both attribute states.** One quantised model rendered from its codebook and
  from its continuous parameters — the comparison the cross-cell figure cannot
  make, because that one contrasts two separately trained runs.

In [ ]:
!python -m tools.chapter_assets \
    --output_root "$DRIVE_ROOT" --out "$ASSETS" --only extras

## 7. The gate — every run against its own evaluation

Each run's recomputed mean is compared with the `eval_metrics.json` that run
wrote during the campaign. Same run, same views, same conventions, so they
should agree to within the render path's tolerance.

**A `MISMATCH` means the model was rendered in a state the campaign did not
evaluate.** Stop and report it rather than using the numbers.

In [ ]:
import csv
rows = list(csv.DictReader(open(ASSETS + '/per_view_selfcheck.csv')))
bad = [r for r in rows if r['verdict'] != 'ok']
print(f"{len(rows)} runs checked, {len(bad)} disagreeing")
print()
print(f"{'cell':5}{'scene':24}{'state':11}{'mine':>8}{'recorded':>10}{'delta':>8}  verdict")
for r in sorted(rows, key=lambda x: (x['cell'], x['scene'])):
    print(f"{r['cell']:5}{r['scene']:24}{r['state']:11}"
          f"{float(r['per_view_mean']):>8.2f}{float(r['eval_metrics']):>10.2f}"
          f"{float(r['delta']):>+8.2f}  {r['verdict']}")
if bad:
    raise SystemExit('Self-check failed. Do not use per_view_metrics.csv for those runs.')

## 8. What landed

In [ ]:
import json, os, collections
man = json.load(open(ASSETS + '/assets_manifest.json'))
print('data products')
for k, v in man['written'].items():
    print(f'  {k:28} {v}')
for k, v in man.get('absent', {}).items():
    print(f'  {k:28} MISSING — {v}')

r = ASSETS + '/renders'
pngs = [p for p in sorted(os.listdir(r)) if p.endswith('.png')]
by_cell = collections.Counter(p.split('_')[1] for p in pngs)
by_kind = collections.Counter(p.split('_', 2)[2][:-4] for p in pngs)
print()
print('renders:', len(pngs), 'images')
print('  by configuration:', dict(sorted(by_cell.items())))
print('  by kind         :', dict(sorted(by_kind.items())))

## 9. Is any scene mean resting on one bad view?

The held-out set is 13 frames, so a scene mean rests on three or four images.
This is the question C1 exists to answer.

In [ ]:
import csv, statistics as st
from collections import defaultdict

rows = list(csv.DictReader(open(ASSETS + '/per_view_metrics.csv')))
by = defaultdict(list)
for r in rows:
    by[(r['cell'], r['scene'])].append(float(r['psnr_pooled']))

print(f"{'cell':5}{'scene':24}{'n':>3}{'mean':>8}{'min':>8}{'max':>8}{'spread':>9}")
for (cell, scene), v in sorted(by.items()):
    if cell not in ('SS', 'A0', 'A2'):
        continue
    print(f'{cell:5}{scene:24}{len(v):>3}{st.mean(v):>8.2f}{min(v):>8.2f}'
          f'{max(v):>8.2f}{max(v) - min(v):>9.2f}')
print()
print('A spread of several dB within one run means the scene mean is a mean over')
print('very unequal views. It does not affect cell-versus-cell comparisons, which')
print('are scored on the same fixed views; it affects reading a scene mean as the')
print('quality on that scene.')

## 10. Bundle and download

In [ ]:
%cd /content
!tar -czf ch4_assets.tar.gz -C /content ch4_assets
import os
print('bundle:', round(os.path.getsize('/content/ch4_assets.tar.gz') / 1e6, 1), 'MB')
from google.colab import files
files.download('/content/ch4_assets.tar.gz')

## 11. Locally

Unpack beside the campaign bundle and rebuild every figure:

```
tar -xzf ch4_assets.tar.gz -C analysis/campaign-2026-09/
python figures/make_chapter4_figures.py     # plots over the data products
python figures/make_chapter4_renders.py     # the qualitative family
```

The first writes the figures that need C1 and C4–C6; the second writes the
per-mechanism comparisons, the overview and the error maps. Both are
deterministic: the same bundle produces byte-identical images.

## Appendix — redoing part of a collection

`--cells` restricts the pass, and a restricted run writes `PARTIAL.txt` so its
output is merged into the full bundle rather than replacing it. Use it when one
group of configurations has to be redone, not for a first collection.

In [ ]:
# Example: the four cells that were added to the render set later.
!python -m tools.chapter_assets \
    --output_root "$DRIVE_ROOT" --out /content/ch4_rest \
    --only C1,renders --cells A0D,A5,A6,A7